# Session 25 — Diffusion Models & Stable Diffusion (Practical)

**Goal:** drive a *pretrained* Stable Diffusion end to end — no training, no building from scratch. We load the finished model and learn to steer it.

We will:
1. Generate our **first image from text** with the 🤗 Diffusers pipeline
2. Play with the **three knobs** — steps, guidance scale, seed — and swap the **scheduler**
3. **Open the pipeline up**: meet the VAE, CLIP text encoder and UNet from the slides, and watch the latents denoise
4. **img2img** — start from an image instead of noise
5. **Inpainting** — regenerate only a masked region
6. **ControlNet** — control composition with a Canny edge map

> ⚠️ **Hardware note:** this notebook needs a GPU with **≥ 8 GB VRAM** (we use fp16 everywhere).
> It is designed for **RunPod** (or Colab with a T4): pick a PyTorch template with CUDA, open Jupyter, upload this notebook, run top to bottom.
> The first run downloads ~4–6 GB of weights from the Hugging Face Hub — subsequent runs use the cache.

---
## Part 0 — Setup

One-time installs, then imports and two tiny helpers we'll reuse all session.

In [ ]:
# Run once per machine (RunPod pods usually start clean)
# diffusers  -> the pipelines           transformers -> CLIP text encoder
# accelerate -> fast/low-RAM loading    safetensors  -> safe weight files
# opencv     -> Canny edges for ControlNet
%pip install -q diffusers transformers accelerate safetensors opencv-python-headless matplotlib

In [ ]:
import torch, gc
from diffusers.utils import load_image
from PIL import Image
import matplotlib.pyplot as plt

# We need a CUDA GPU for this session
assert torch.cuda.is_available(), "No GPU found — on RunPod pick a CUDA/PyTorch template"
device = "cuda"
dtype  = torch.float16          # half precision: 2x smaller, plenty accurate for inference

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
def show(images, titles=None, size=4):
    """Display one image or a row of images side by side."""
    if isinstance(images, Image.Image):
        images = [images]
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    axes = [axes] if len(images) == 1 else axes
    for ax, im in zip(axes, images):
        ax.imshow(im); ax.axis("off")
    if titles:
        for ax, t in zip(axes, titles):
            ax.set_title(t, fontsize=10)
    plt.tight_layout(); plt.show()

def free_vram(*names):
    """Delete pipelines/models by NAME and clear the CUDA cache between parts.
    SD components are big — this keeps us inside 8 GB. Safe to call twice."""
    g = globals()
    for n in names:
        if n in g:
            del g[n]
    gc.collect()
    torch.cuda.empty_cache()
    print("VRAM in use:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

---
## Part 1 — Your First Generation (text → image)

One pipeline object bundles all three specialists from the slides — **CLIP text encoder + UNet/scheduler + VAE decoder** — pre-wired and pretrained.

We use **Stable Diffusion v1.5**, the classic 512×512 checkpoint (the `sd-legacy` repo is its official home on the Hub).

In [ ]:
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",   # ~4 GB download on first run
    torch_dtype=dtype,
    safety_checker=None,                 # saves ~1 GB VRAM in class; keep it in real apps
).to(device)

# Optional safety nets on small GPUs (slightly slower, much lighter):
# pipe.enable_attention_slicing()
# pipe.enable_model_cpu_offload()     # use INSTEAD of .to(device)

In [ ]:
prompt = "a cozy wooden cabin in a snowy forest at golden hour, cinematic, highly detailed"

generator = torch.Generator(device).manual_seed(42)   # the seed knob -> reproducible noise

image = pipe(
    prompt,
    num_inference_steps=30,     # knob 1: how many denoising steps
    guidance_scale=7.5,         # knob 2: how strongly to follow the prompt (CFG)
    generator=generator,        # knob 3: which starting noise
).images[0]

show(image, ["30 steps · CFG 7.5 · seed 42"])

**Checkpoint — connect to the theory:** that progress bar you just watched *is* the reverse process from Part 1 of the slides: 30 UNet calls, each predicting the noise ε̂ in the current 4×64×64 latent, the scheduler subtracting it — then one single VAE decode at the end.

▶ **Try it:** re-run the cell with the *same* seed (identical image — DDIM-style determinism), then change the seed (same idea, new image).

---
## Part 2 — The Three Knobs & the Scheduler

Same prompt, same seed — we vary **one parameter at a time** and read the effect.
(Each loop below re-runs the pipeline a few times, so expect a few minutes total.)

In [ ]:
# Knob 1: number of denoising steps — watch quality saturate
step_counts = [5, 10, 20, 40]
imgs = [
    pipe(prompt, num_inference_steps=n, guidance_scale=7.5,
         generator=torch.Generator(device).manual_seed(42)).images[0]
    for n in step_counts
]
show(imgs, [f"{n} steps" for n in step_counts], size=3.2)
# Expect: 5 = mushy shapes, 10 = almost there, 20-40 = clean; gains flatten out.

In [ ]:
# Knob 2: guidance scale (classifier-free guidance)
# Low = model free-styles; high = obeys the prompt too hard (burnt colors)
cfgs = [1.5, 7.5, 15]
imgs = [
    pipe(prompt, num_inference_steps=30, guidance_scale=g,
         generator=torch.Generator(device).manual_seed(42)).images[0]
    for g in cfgs
]
show(imgs, [f"CFG {g}" for g in cfgs], size=3.2)
# Expect: 1.5 = vague/off-topic, 7.5 = sweet spot, 15 = oversaturated, "over-obeying".

In [ ]:
# Negative prompt: steer AWAY from things (same CFG mechanism, contrasting
# against your negative text instead of an empty prompt)
image = pipe(
    prompt,
    negative_prompt="blurry, low quality, deformed, watermark, text",
    num_inference_steps=30, guidance_scale=7.5,
    generator=torch.Generator(device).manual_seed(42),
).images[0]
show(image, ["with negative prompt"])

In [ ]:
# Swap the scheduler = swap the sampler (the DDIM legacy from the slides).
# DPM-Solver++ gives clean images in ~20 steps instead of 30-50.
from diffusers import DPMSolverMultistepScheduler

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

image = pipe(prompt, num_inference_steps=20, guidance_scale=7.5,
             generator=torch.Generator(device).manual_seed(42)).images[0]
show(image, ["DPM-Solver++ · only 20 steps"])

▶ **Exercise 2.1:** find the *lowest* step count where DPM-Solver++ still gives you an acceptable cabin. 8? 12? 15?

▶ **Exercise 2.2:** fix the seed and change exactly one word of the prompt (`snowy` → `autumn`). Because the starting noise is identical, you get a controlled experiment on the prompt.

---
## Part 3 — Opening the Pipeline: VAE, CLIP, UNet

The pipeline is just a container. Let's meet the three specialists from the slides — and check the tensor shapes we memorised: **77×768** (text), **4×64×64** (latent), **3×512×512** (pixels).

In [ ]:
# The components, by name
def params(m):
    return f"{sum(p.numel() for p in m.parameters())/1e6:.0f}M params"

print(f"{type(pipe.text_encoder).__name__:30s} {params(pipe.text_encoder):>12s}   (CLIP text encoder)")
print(f"{type(pipe.unet).__name__:30s} {params(pipe.unet):>12s}   (the noise predictor)")
print(f"{type(pipe.vae).__name__:30s} {params(pipe.vae):>12s}   (encoder + decoder)")
print(f"{type(pipe.scheduler).__name__:30s} {'0 params':>12s}   (just the sampling maths)")

In [ ]:
# 1) The prompt really becomes 77 tokens x 768 numbers
tokens = pipe.tokenizer(prompt, padding="max_length",
                        max_length=77, return_tensors="pt")
with torch.no_grad():
    text_emb = pipe.text_encoder(tokens.input_ids.to(device))[0]
print("token ids :", tokens.input_ids.shape)     # [1, 77]
print("embeddings:", text_emb.shape)             # [1, 77, 768]  <- the conditioning matrix

In [ ]:
# 2) The VAE round trip: image -> 4x64x64 latent -> image
test_img = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/img2img-init.png"
).resize((512, 512))

import numpy as np
x = (torch.from_numpy(np.array(test_img)).float() / 127.5 - 1)   # to [-1, 1]
x = x.permute(2, 0, 1).unsqueeze(0).to(device, dtype)            # [1, 3, 512, 512]

with torch.no_grad():
    latent = pipe.vae.encode(x).latent_dist.sample() * pipe.vae.config.scaling_factor
    recon  = pipe.vae.decode(latent / pipe.vae.config.scaling_factor).sample

print("pixels :", tuple(x.shape), "->", "latent:", tuple(latent.shape), " (48x fewer numbers)")
recon_img = ((recon[0].float().cpu().permute(1, 2, 0) + 1) * 127.5).clamp(0, 255).numpy().astype("uint8")
show([test_img, Image.fromarray(recon_img)], ["original", "VAE encode → decode"])
# The reconstruction should be near-identical: compression, not generation.

In [ ]:
# 3) Watch the latents denoise — the 'image emerging' strip from the slides
snapshots = []

def grab(pipe_ref, step, timestep, kwargs):
    if step % 6 == 0:                       # every 6th step
        lat = kwargs["latents"]
        with torch.no_grad():
            img = pipe_ref.vae.decode(lat / pipe_ref.vae.config.scaling_factor).sample
        img = ((img[0].float().cpu().permute(1, 2, 0) + 1) * 127.5).clamp(0, 255).numpy().astype("uint8")
        snapshots.append((step, Image.fromarray(img)))
    return kwargs

_ = pipe(prompt, num_inference_steps=30, guidance_scale=7.5,
         generator=torch.Generator(device).manual_seed(42),
         callback_on_step_end=grab)

show([s[1] for s in snapshots], [f"step {s[0]}" for s in snapshots], size=2.6)
# Coarse-to-fine: color fields -> composition -> texture. Layout locks in EARLY.

---
## Part 4 — img2img: Start From an Image, Not From Noise

Mechanism (slide 19): **VAE-encode** your image → **add partial noise** with the forward-process shortcut → **denoise from there** with your prompt.

`strength` = how far into the noise schedule you jump: `0.0` returns your image, `1.0` ignores it.

> 💡 `AutoPipeline...from_pipe(pipe)` **reuses the weights already on the GPU** — no re-download, no extra VRAM.

In [ ]:
from diffusers import AutoPipelineForImage2Image

img2img = AutoPipelineForImage2Image.from_pipe(pipe)   # shares components with `pipe`

init_image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/img2img-init.png"
).resize((512, 512))

i2i_prompt = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"

results = [
    img2img(i2i_prompt, image=init_image, strength=s, guidance_scale=7.5,
            generator=torch.Generator(device).manual_seed(7)).images[0]
    for s in (0.35, 0.6, 0.9)
]
show([init_image] + results, ["input", "strength 0.35", "strength 0.6", "strength 0.9"], size=3.2)
# 0.35: composition intact, style shifting - 0.6: prompt's world, poses persist - 0.9: input almost gone.

▶ **Exercise 4.1:** upload a photo or drawing of your own (`init_image = Image.open("my_file.png")...`) and restyle it. Find your sweet spot — for most restyling jobs it lives around **0.5–0.7**.

---
## Part 5 — Inpainting: Edit Only Where the Mask Says

Three inputs: image + **mask** (white = regenerate, black = keep) + prompt.
Every denoising step blends generated latents (inside the mask) with re-noised original latents (outside) — that's why the seam is invisible.

We use the **dedicated inpainting checkpoint** — same SD 1.5, fine-tuned to take the mask as extra UNet input.

In [ ]:
# The inpainting checkpoint is a DIFFERENT set of weights -> free the old ones first
free_vram("img2img")   # keeps `pipe` for now; below 10 GB VRAM? drop it too: free_vram("pipe")

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

inpaint = StableDiffusionInpaintPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-inpainting",
    torch_dtype=dtype, safety_checker=None,
).to(device)

img_url  = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint.png"
mask_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint_mask.png"
image = load_image(img_url).resize((512, 512))
mask  = load_image(mask_url).resize((512, 512))

result = inpaint(
    prompt="black cat with glowing eyes, cute, adorable, highly detailed, 8k",
    image=image, mask_image=mask,
    num_inference_steps=30, guidance_scale=7.5,
    generator=torch.Generator(device).manual_seed(11),
).images[0]

show([image, mask, result], ["original", "mask (white = repaint)", "result"], size=3.4)
# Same road, same trees, same sky - the mountain is now a cat. Photoshop's
# Generative Fill is exactly this operation.

▶ **Exercise 5.1:** draw your own mask — quickest way: create a black 512×512 image in any editor, paint the region white, upload both. Or build one in code with PIL's `ImageDraw.rectangle`.

▶ **Think ahead (course capstone):** the mask doesn't have to be hand-drawn — a segmentation model can make it. *Detect → segment → inpaint* is the GenAI pipeline this phase builds toward.

---
## Part 6 — ControlNet: Show, Don't Tell

Prompts can't specify exact geometry. ControlNet adds one extra conditioning image — here **Canny edges** — through a small trained side-network, while the original SD weights stay **frozen** (slide 21).

In [ ]:
# ControlNet loads the full SD pipeline + a ~1.4 GB side-network -> clean up first
free_vram("inpaint", "pipe")

In [ ]:
import cv2
import numpy as np
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline

# 1) Make the condition: Canny edge map of a reference photo
ref = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/img2img-init.png"
).resize((512, 512))
edges = cv2.Canny(np.array(ref), 100, 200)
edge_image = Image.fromarray(np.stack([edges] * 3, axis=2))   # to 3-channel

# 2) Load ControlNet (canny flavor) + SD 1.5 around it
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=dtype)
cn_pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",
    controlnet=controlnet, torch_dtype=dtype, safety_checker=None,
).to(device)

In [ ]:
# 3) Same edges, three different prompts -> layout locked, content free
prompts = [
    "toy astronauts made of white porcelain, studio lighting",
    "bronze statues of astronauts, museum photography",
    "astronauts made of colorful lego bricks, product photo",
]
results = [
    cn_pipe(pr, image=edge_image, num_inference_steps=25, guidance_scale=7.5,
            generator=torch.Generator(device).manual_seed(3)).images[0]
    for pr in prompts
]
show([ref, edge_image] + results,
     ["reference", "canny edges", "porcelain", "bronze", "lego"], size=2.7)
# Every output obeys the SAME outline - the prompt only decides what fills it.

▶ **Exercise 6.1:** run Canny on your own photo and generate 3 style variations.

▶ **Exercise 6.2 (explore):** other ControlNet flavors are drop-in replacements — try `lllyasviel/sd-controlnet-depth` (needs a depth map) or `lllyasviel/sd-controlnet-openpose` (human pose). The `controlnet_aux` package generates those condition maps for you.

---
## Wrap-up

You drove the entire Session 25 stack without training anything:

| You did | Theory it proved |
|---|---|
| `pipe(prompt, steps, guidance, generator)` | reverse process + CFG + seeded noise |
| scheduler swap → 20 steps | DDIM legacy: samplers are exchangeable |
| tokenizer → `[1, 77, 768]` | CLIP conditioning matrix |
| VAE round trip → `[1, 4, 64, 64]` | 48× latent compression |
| step-by-step snapshots | coarse-to-fine denoising |
| img2img `strength` | partial forward noising as a starting point |
| inpainting mask | per-step latent blending |
| ControlNet edges | frozen base + trained side-network |

**Homework:** bring one prompt you loved and one that failed — we'll debug it with today's vocabulary.

**Next session:** LoRA & model customisation — DreamBooth and textual inversion: teaching this frozen giant *your* concepts by training something tiny on the side.